# Examining MS SQL (TDS) traffic with Zeek

This is the hunt from [*Threat Hunting with Python Part 4: Examining Microsoft SQL Based Historian Traffic*](../docs/2018-03-threat-hunting-with-python-part-4-mssql-historian.md), redone with Zeek instead of tshark. The original notebook in this folder, `tds.ipynb`, drives tshark through pyshark and picks fields out of each packet; it still runs and is kept as written.

Here the same capture goes through Zeek with the [zeek-tds](https://github.com/dgunter/zeek-tds) analyzer, which understands the Tabular Data Stream protocol end to end: it reassembles messages across packets, resolves the numbered procedures drivers use for prepared statements, renders every parameter value by type, and writes logs that join to the rest of Zeek's output. [ParseZeekLogs](https://github.com/dgunter/ParseZeekLogs) then reads those logs into pandas. Zeek runs in Docker so nothing but Docker is needed on the machine; the same image is what you would run on a sensor.

In [1]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import pandas as pd
from parsezeeklogs import ZeekLog

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PCAP = Path("ms-sql-tds-rpc-requests.pcap").resolve()
IMAGE = "zeek-tds:0.2.0"

## Run Zeek over the capture

The image is Zeek 8.2 with the zeek-tds package installed; `zeek/Dockerfile` builds it in a few minutes if it is not already present.

In [2]:
def ensure_image():
    if subprocess.run(["docker", "image", "inspect", IMAGE], capture_output=True).returncode != 0:
        subprocess.run(
            ["docker", "build", "-q", "-t", IMAGE, str(Path("zeek").resolve())], check=True
        )


def run_zeek(pcap: Path, *options: str) -> Path:
    out = Path(tempfile.mkdtemp(prefix="zeek-tds-"))
    subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{pcap.parent}:/pcap:ro",
            "-v",
            f"{out}:/data",
            "-u",
            "0",
            IMAGE,
            "-r",
            f"/pcap/{pcap.name}",
            *options,
        ],
        check=True,
        capture_output=True,
    )
    return out


ensure_image()
logs = run_zeek(PCAP)
sorted(p.name for p in logs.glob("*.log"))

['conn.log',
 'packet_filter.log',
 'tds.log',
 'tds_rpc.log',
 'tds_sql_batch.log']

## Load the logs

Every Zeek log becomes a typed DataFrame. The `uid` column is the connection identifier that ties the TDS logs to `conn.log` and to each other.

In [3]:
def load(name: str, where: Path | None = None) -> pd.DataFrame:
    with ZeekLog((where or logs) / f"{name}.log") as log:
        df = pd.DataFrame(log)
    if "ts" in df:
        df["ts"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    return df


conn = load("conn")
tds = load("tds")
batches = load("tds_sql_batch")
rpc = load("tds_rpc")
print(
    f"{len(conn)} connections, {len(tds)} TDS messages, "
    f"{len(batches)} SQL batches, {len(rpc)} procedure calls"
)
conn[
    [
        "ts",
        "uid",
        "id.orig_h",
        "id.orig_p",
        "id.resp_h",
        "id.resp_p",
        "service",
        "orig_bytes",
        "resp_bytes",
    ]
]

12 connections, 29 TDS messages, 3 SQL batches, 17 procedure calls


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,service,orig_bytes,resp_bytes
0,2009-04-28 00:18:37.888015032+00:00,CFgl432RiV5QMf8kWl,10.111.111.111,1111,10.0.0.1,1433,None,482.0,392.0
1,2009-12-02 14:00:00.004436970+00:00,CNYwCO2ekLU4aDsC65,10.111.111.111,2222,10.0.0.1,1433,None,44.0,17.0
2,2009-12-02 14:00:00.022561073+00:00,CVNYx24sE1cSVevXz4,10.111.111.111,3333,10.0.0.1,1433,None,NaN,NaN
3,2009-12-02 14:00:00.033700943+00:00,CvX7ZcsIuoRn5WDc8,10.111.111.111,4444,10.0.0.1,1433,None,NaN,NaN
4,2009-12-02 14:00:00.716203928+00:00,CkVlKs11iDPrCFBXl3,10.111.111.111,5555,10.0.0.1,1433,None,994.0,1143.0
5,2009-12-02 14:01:14.884130955+00:00,CRZiJOln0tcFoRmg,10.111.111.111,6666,10.0.0.1,1433,None,8339.0,0.0
6,2009-12-02 14:01:17.536189079+00:00,CSKq7A1JndbWUM1xT1,10.111.111.111,7777,10.0.0.1,1433,None,NaN,NaN
7,2009-12-02 14:01:22.456089973+00:00,CerKLf1w3qoFTVOCnb,10.111.111.111,8888,10.0.0.1,1433,None,NaN,NaN
8,2010-07-02 11:00:44.666074991+00:00,CMrmlKeYGyZvEOqIe,10.111.111.111,33333,10.0.0.1,1433,None,NaN,NaN
9,2010-07-02 11:00:44.650715113+00:00,CHNGko1juvnYRD1XQh,10.111.111.111,22222,10.0.0.1,1433,None,NaN,NaN


## What kinds of messages, and which way

`tds.log` is the skeleton: one line per message after reassembly. The original notebook counted TDS packet types; here the counts are per message, and responses carry how many rows they returned.

In [4]:
tds["direction"] = tds["is_orig"].map({True: "client", False: "server"})
tds.groupby(["direction", "msg_type"]).agg(
    messages=("uid", "size"), bytes=("len", "sum"), packets=("packets", "sum"), rows=("rows", "sum")
)

messages  bytes  packets  rows
direction msg_type                                      
client    rpc                   16  12030       17   0.0
          sql_batch              3    400        3   0.0
server    tabular_result        10   1472       10   7.0

## What they typed: SQL batches

In [5]:
batches[["ts", "id.orig_p", "transaction_descriptor", "query"]]

,ts,id.orig_p,transaction_descriptor,query
0,2009-04-28 00:18:37.888015032+00:00,1111,0,set transaction isolation level read committed set implicit_transactions off
1,2009-12-02 14:00:00.004436970+00:00,2222,0,COMMIT TRANSACTION
2,2009-12-02 14:00:00.716203928+00:00,5555,0,set transaction isolation level read committed set implicit_transactions off


## What they called: procedures

This is where the 2018 analysis ran out of road. Drivers ship prepared statements as calls to `sp_prepexec` and `sp_execute`, addressed by number on the wire; the original notebook mapped the numbers to names by hand and never saw the SQL inside them. Zeek resolves the procedure, lifts the SQL text into `statement`, and keeps the handle that ties an `sp_execute` back to the `sp_prepexec` that prepared it.

In [6]:
rpc["procedure"].value_counts()

procedure
sp_prepexec                             5
sp_execute                              3
proc_GetMyExampleTableSampleMetaData    2
p_GetBogusData                          1
sp_executesql                           1
p_SaveExample                           1
p_SetBogusSample                        1
p_GetMyExampleTableRowCount             1
dbo.proc_GetMySampleDataItems           1
proc_FetchMyExampleData                 1
Name: count, dtype: int64

In [7]:
rpc[["ts", "id.orig_p", "procedure", "proc_id", "handle", "statement"]]

,ts,id.orig_p,procedure,proc_id,handle,statement
0,2009-04-28 00:18:37.918653011+00:00,1111,sp_prepexec,13.0,0,select * from test_table_1 where name = @P0 and id = @P1
1,2009-12-02 14:00:00.747371912+00:00,5555,sp_prepexec,13.0,0,"create table newsyb (column1 char(30) not null, column2 char(30) null,column3 char(30) null)"
2,2009-12-02 14:00:01.226156950+00:00,5555,sp_prepexec,13.0,0,"insert INTO newsyb (column1, column2, column3) VALUES ('first', 'second', 'third')"
3,2009-12-02 14:00:01.711920977+00:00,5555,sp_execute,12.0,2,NaN
4,2009-12-02 14:00:01.711920977+00:00,5555,sp_execute,12.0,2,NaN
5,2009-12-02 14:00:03.727473021+00:00,5555,sp_prepexec,13.0,0,select * from newsyb
6,2009-12-02 14:00:05.827626944+00:00,5555,sp_execute,12.0,3,NaN
7,2009-12-02 14:00:07.864718914+00:00,5555,sp_prepexec,13.0,0,drop table newsyb
8,2009-12-02 14:00:00.022561073+00:00,3333,p_GetBogusData,NaN,NaN,NaN
9,2009-12-02 14:00:00.033700943+00:00,4444,sp_executesql,NaN,NaN,"SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleTable].[EntityHistoryI..."


Every parameter is rendered by type, so the historian's own procedures show what they were asked for:

In [8]:
for _, row in rpc[~rpc["procedure"].str.startswith("sp_")].iterrows():
    print(f"{row['procedure']}")
    for p in row["parameters"]:
        print(f"    {p[:110]}")

p_GetBogusData
    @SearchType intn(1) = 1
    @MaxWaitTimeInSeconds intn(4) = 0
    @ProcessNegativeAck intn(1) = 0
p_SaveExample
    @LongParam nvarchar(max) = Studenckie Koło Przewodników Turystycznych w Gdańsku
zaprasza na:
XXXV Nocne Marsze
    @Operation intn(4) = 1
p_SetBogusSample
    @BogusDetailsID intn(8) = 74565
    @BogusStatusID intn(8) = 5
    @ResultCode nvarchar(1) = NULL
    @ResultMsg nvarchar(1) = NULL
    @ErrorCode nvarchar(1) = NULL
    @ErrorMsg nvarchar(1) = NULL
    @ExampleBogusGeneratedID nvarchar(1) = NULL
    @ExampleType intn(4) = 1
p_GetMyExampleTableRowCount
dbo.proc_GetMySampleDataItems
    @SampleItemId uniqueidentifier = 4EC31A66-A214-4853-A77E-E7060FFFFF07
    @DataId nvarchar(1) = NULL
    @DataItemType uniqueidentifier = BCB9459B-83A8-4564-B1D3-E9E198478F4E
    @TableId nvarchar(1) = NULL
    @MaxFetchSize intn(4) = 100
    @SomeOtherSampleId intn(4) = 0
proc_FetchMyExampleData
    @p1 uniqueidentifier = 67452301-AB89-EFCD-0123-456789ABCDEF
    @p

## The questions from the post

The post looked for two specific calls. Both are a filter away, and now the parameters come with them.

In [9]:
rpc.loc[
    rpc["procedure"] == "p_GetBogusData",
    ["ts", "id.orig_h", "id.resp_h", "procedure", "parameters"],
]

,ts,id.orig_h,id.resp_h,procedure,parameters
8,2009-12-02 14:00:00.022561073+00:00,10.111.111.111,10.0.0.1,p_GetBogusData,"[@SearchType intn(1) = 1, @MaxWaitTimeInSeconds intn(4) = 0, @ProcessNegativeAck intn(1) = 0]"


In [10]:
rpc.loc[
    rpc["procedure"].isin(["p_SetBogusSample", "sp_executesql"]),
    ["ts", "id.orig_h", "id.resp_h", "procedure", "statement", "parameters"],
]

,ts,id.orig_h,id.resp_h,procedure,statement,parameters
9,2009-12-02 14:00:00.033700943+00:00,10.111.111.111,10.0.0.1,sp_executesql,"SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleTable].[EntityHistoryI...","[@p1 nvarchar(467) = SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleT..."
11,2009-12-02 14:01:17.536189079+00:00,10.111.111.111,10.0.0.1,p_SetBogusSample,NaN,"[@BogusDetailsID intn(8) = 74565, @BogusStatusID intn(8) = 5, @ResultCode nvarchar(1) = NULL, @ResultMsg nvarchar(1)..."


## What tshark did not see

The original notebook's pyshark pass found 8 named procedures and 7 numbered ones. Zeek logs 17 procedure calls from the same 16 RPC messages. The two differences are structural, not a matter of field extraction:

- **Messages that span packets.** One RPC (`p_SaveExample`, with an 8 KB `nvarchar(max)` argument) arrives in two TDS packets. tshark dissects packet by packet and does not reassemble TDS messages, so it saw two RPC frames with no procedure in either. Zeek reassembles the message first.
- **Messages that hold several calls.** One 53-byte message carries two `sp_execute` calls separated by a batch flag. tshark reports the first procedure per packet; Zeek logs each call.

Both cases are visible in the Zeek output: RPC messages with more than one packet, and timestamps at which more than one procedure call was logged.

In [11]:
multi_packet = tds[(tds["msg_type"] == "rpc") & (tds["packets"] > 1)][
    ["ts", "id.orig_p", "len", "packets"]
]
multi_call = rpc.groupby("ts").size()
multi_call = multi_call[multi_call > 1]
print("RPC messages spanning several packets:")
print(multi_packet.to_string(index=False))
print("\nTimestamps with more than one procedure call in one message:")
print(
    rpc[rpc["ts"].isin(multi_call.index)][["ts", "id.orig_p", "procedure", "parameters"]].to_string(
        index=False
    )
)

RPC messages spanning several packets:
                                 ts  id.orig_p  len  packets
2009-12-02 14:01:14.884730101+00:00       6666 8323        2

Timestamps with more than one procedure call in one message:
                                 ts  id.orig_p  procedure        parameters
2009-12-02 14:00:01.711920977+00:00       5555 sp_execute [@p1 intn(4) = 2]
2009-12-02 14:00:01.711920977+00:00       5555 sp_execute [@p1 intn(4) = 2]


## The other logs, on today's traffic

The 2009 capture has no logins in it, so three of the analyzer's logs stay empty. The zeek-tds test corpus, recorded against Azure SQL Edge (the SQL Server 2019 engine) with Python, FreeTDS and .NET clients, exercises the rest. The captures are small and public; this fetches them from the tagged release.

In [12]:
import urllib.request

CORPUS = "https://raw.githubusercontent.com/dgunter/zeek-tds/v0.2.0/testing/Traces"
TRACES = [
    "sqledge-pytds-workload",
    "sqledge-pytds-failed-login",
    "sqledge-freetds",
    "sqledge-pytds-ntlm",
    "sqledge-dotnet-mars",
    "ssrp-browser",
]
captures = Path(tempfile.mkdtemp(prefix="zeek-tds-corpus-"))
for name in TRACES:
    urllib.request.urlretrieve(f"{CORPUS}/{name}.pcap", captures / f"{name}.pcap")

runs = {
    name: run_zeek(captures / f"{name}.pcap", "TDS::log_results=T", "TDS::result_sample_rows=3")
    for name in TRACES
}
{
    name: sorted(
        p.name for p in out.glob("*.log") if p.name not in ("packet_filter.log", "conn.log")
    )
    for name, out in runs.items()
}

{'sqledge-pytds-workload': ['tds.log',
  'tds_error.log',
  'tds_login.log',
  'tds_result.log',
  'tds_rpc.log',
  'tds_sql_batch.log'],
 'sqledge-pytds-failed-login': ['tds.log', 'tds_error.log', 'tds_login.log'],
 'sqledge-freetds': ['ssl.log',
  'tds.log',
  'tds_login.log',
  'tds_result.log',
  'tds_rpc.log',
  'tds_sql_batch.log',
  'x509.log'],
 'sqledge-pytds-ntlm': ['ntlm.log',
  'tds.log',
  'tds_error.log',
  'tds_login.log'],
 'sqledge-dotnet-mars': ['ssl.log',
  'tds.log',
  'tds_login.log',
  'tds_result.log',
  'tds_rpc.log',
  'tds_sql_batch.log',
  'x509.log'],
 'ssrp-browser': ['ssrp.log']}

### tds_login.log: who connected, with what

One line per connection: the client's host name, account, application and driver from LOGIN7, the encryption both sides asked for, the server's product and version from its answer, and whether the login succeeded. The failed-login capture shows the other outcome: error 18456 recorded on the same line.

In [13]:
login_cols = [
    "hostname",
    "username",
    "has_password",
    "password_in_clear",
    "integrated_auth",
    "app_name",
    "library",
    "database",
    "tds_version",
    "client_encryption",
    "server_encryption",
    "encrypted",
    "server_product",
    "server_product_version",
    "success",
    "error_number",
    "error_message",
]
logins = pd.concat(
    [
        load("tds_login", runs[n]).assign(capture=n)
        for n in TRACES
        if (runs[n] / "tds_login.log").exists()
    ]
)
logins.set_index("capture")[login_cols].T

capture,sqledge-pytds-workload,sqledge-pytds-failed-login,sqledge-freetds,sqledge-pytds-ntlm,sqledge-dotnet-mars
hostname,workstation-01,workstation-01,None,workstation-01,None
username,sa,sa,None,,None
has_password,True,True,None,False,None
password_in_clear,True,True,False,False,False
integrated_auth,False,False,None,True,None
app_name,zeek-tds-corpus,zeek-tds-badpw,None,zeek-tds-ntlm,None
library,Python TDS Library,Python TDS Library,None,Python TDS Library,None
database,zeektds,master,None,master,None
tds_version,7.4,7.4,None,7.4,None
client_encryption,not_supported,not_supported,off,not_supported,off


### tds_error.log: what the server refused

Every ERROR token, with number, severity and text. 208 is the "invalid object name" that reconnaissance trips; 18456 is a failed login; 18452 is a Windows login the server would not trust.

In [14]:
errors = pd.concat(
    [
        load("tds_error", runs[n]).assign(capture=n)
        for n in TRACES
        if (runs[n] / "tds_error.log").exists()
    ]
)
errors[["capture", "number", "class", "state", "message", "server_name"]]

,capture,number,class,state,message,server_name
0,sqledge-pytds-workload,208,16,1,Invalid object name 'dbo.does_not_exist'.,sqledge01
1,sqledge-pytds-workload,50000,16,1,custom failure 42,sqledge01
0,sqledge-pytds-failed-login,18456,14,1,Login failed for user 'sa'.,sqledge01
0,sqledge-pytds-ntlm,18452,14,1,Login failed. The login is from an untrusted domain and cannot be used with Integrated authentication.,sqledge01


### Encrypted sessions: ssl.log and ntlm.log under the same uid

FreeTDS encrypts only the login by default. The TLS handshake travels inside TDS PRELOGIN packets; the analyzer unwraps it and hands it to Zeek's SSL analyzer, so `conn.log` shows both services and `ssl.log` has the cipher and certificate for the SQL session. A Windows login does the same with NTLM.

In [15]:
for name in ("sqledge-freetds", "sqledge-pytds-ntlm"):
    c = load("conn", runs[name])
    print(f"{name}: conn.log service = {c['service'].iloc[0]}, uid = {c['uid'].iloc[0]}")
    for extra in ("ssl", "ntlm"):
        if (runs[name] / f"{extra}.log").exists():
            df = load(extra, runs[name])
            cols = [
                col
                for col in (
                    "uid",
                    "version",
                    "cipher",
                    "curve",
                    "server_name",
                    "established",
                    "hostname",
                    "domainname",
                    "username",
                )
                if col in df
            ]
            print(df[cols].to_string(index=False))
    print()

sqledge-freetds: conn.log service = tds,ssl, uid = CePJz11u2dS55ptDUe
               uid version                                cipher  curve server_name  established
CePJz11u2dS55ptDUe  TLSv12 TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256 x25519        None         True

sqledge-pytds-ntlm: conn.log service = tds,ntlm, uid = Ckca431lidT25yN8Cj
               uid hostname domainname username
Ckca431lidT25yN8Cj     None       None     None



### MARS: several sessions on one connection

A .NET client with `MultipleActiveResultSets=True` multiplexes sessions over one TCP connection. The analyzer unwraps the SMP framing and `tds.log` records the session id, so interleaved requests and their responses can be paired.

In [16]:
mars = load("tds", runs["sqledge-dotnet-mars"])
mars[["ts", "is_orig", "msg_type", "len", "session", "rows", "row_count"]].iloc[3:15]

,ts,is_orig,msg_type,len,session,rows,row_count
3,2026-09-06 00:06:55.269366026+00:00,True,rpc,233,1.0,NaN,NaN
4,2026-09-06 00:06:55.288081884+00:00,False,tabular_result,1983,1.0,45.0,45.0
5,2026-09-06 00:06:55.291708946+00:00,True,rpc,212,2.0,NaN,NaN
6,2026-09-06 00:06:55.297415972+00:00,False,tabular_result,51,2.0,1.0,1.0
7,2026-09-06 00:06:55.299412012+00:00,True,rpc,212,2.0,NaN,NaN
8,2026-09-06 00:06:55.299654007+00:00,False,tabular_result,51,2.0,1.0,1.0
9,2026-09-06 00:06:55.299787045+00:00,True,rpc,212,2.0,NaN,NaN
10,2026-09-06 00:06:55.299958944+00:00,False,tabular_result,51,2.0,1.0,1.0
11,2026-09-06 00:06:55.300045013+00:00,True,rpc,212,2.0,NaN,NaN
12,2026-09-06 00:06:55.300224066+00:00,False,tabular_result,51,2.0,1.0,1.0


### tds_result.log: what came back

Opt-in, because it describes the data itself. Column names and types and the row count per result set; with `TDS::result_sample_rows` set, the first rows rendered by type.

In [17]:
results = load("tds_result", runs["sqledge-pytds-workload"])
results[["columns", "types", "rows", "sample"]].head(6)

,columns,types,rows,sample
0,"[id, tag, value, quality, ts, note, blob, amount, price, flag, guid, d, t, dto, big, xmlcol]","[int, nvarchar(64), floatn(8), intn(1), datetime2, varchar(200), varbinary(max), decimaln(12,4), moneyn(8), bitn, un...",60,[1|PUMP-000.FLOW|NULL|NULL|2026-09-05 12:00:00.000|NULL|0x01|1234.5678|99.9900|false|09BBB7AA-CE65-4CCF-803A-D2CE6C5...
1,[],[intn(4)],1,[45]
2,"[id, tag, value]","[int, nvarchar(64), floatn(8)]",5,"[10|PUMP-009.FLOW|21.5, 9|PUMP-008.FLOW|20.5, 8|PUMP-007.FLOW|NULL]"
3,"[big, x]","[nvarchar(max), xml]",1,[xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...


### ssrp.log: how clients and scanners find instances

Before the first TDS packet, a client that knows only an instance name asks the SQL Server Browser service on UDP 1434. So does every scanner. The reply is an inventory: each instance, its build and its port.

In [18]:
load("ssrp", runs["ssrp-browser"])[["request", "instance", "instances", "dac_port", "answered"]]

,request,instance,instances,dac_port,answered
0,enumerate,NaN,"[SQLEDGE01\MSSQLSERVER 15.0.2000.1574 tcp/1433, SQLEDGE01\HISTORIAN 14.0.3456.2 tcp/1533]",NaN,True
1,instance,HISTORIAN,[SQLEDGE01\HISTORIAN 14.0.3456.2 tcp/1533],NaN,True
2,instance,MSSQLSERVER,[SQLEDGE01\MSSQLSERVER 15.0.2000.1574 tcp/1433],NaN,True
3,dac,HISTORIAN,None,1534.0,True
4,broadcast,NaN,"[SQLEDGE01\MSSQLSERVER 15.0.2000.1574 tcp/1433, SQLEDGE01\HISTORIAN 14.0.3456.2 tcp/1533]",NaN,True


### Notices

The package's detection script turns the common questions into Zeek notices. With default thresholds nothing fires on this corpus; here the large-result threshold is lowered to fifty rows and the brute-force threshold to one failure to show two of them.

In [19]:
demo = run_zeek(
    captures / "sqledge-pytds-workload.pcap",
    "TDS::large_result_rows=50",
    "TDS::detect_cleartext_password=T",
)
demo_fail = run_zeek(captures / "sqledge-pytds-failed-login.pcap", "TDS::bruteforce_threshold=1.0")
notices = pd.concat([load("notice", d) for d in (demo, demo_fail) if (d / "notice.log").exists()])
notices[["note", "msg"]]

,note,msg
0,TDS::Cleartext_Password,172.19.0.1 sent a SQL Server password outside TLS to 172.19.0.2 as sa
1,TDS::Large_Result,"172.19.0.1 received a 60-row, 12995-byte result from 172.19.0.2"
0,TDS::Login_Bruteforce,172.19.0.1 failed 1 SQL Server logins in 10.0 mins against 1 server(s)


## Where this goes next

On a sensor the analyzer runs continuously, and these logs accumulate for every SQL Server session on the network: who connected with which driver, what they ran, what the server refused, how much came back, and, through `ssl.log` and `ntlm.log`, the certificate and the Windows account behind it. The 2009 capture predates TLS on the wire; on a modern network encrypted sessions keep only the login negotiation and the certificate, which is why the plaintext decoding matters most where this series started, on control-system networks.

In [20]:
for d in [logs, captures, demo, demo_fail, *runs.values()]:
    shutil.rmtree(d, ignore_errors=True)